<a href="https://colab.research.google.com/github/sarahj78/Prodigiorum-portentorum/blob/main/pyOdysseus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Installation**

In [1]:
!python --version

Python 3.12.13


In [2]:
!git clone https://github.com/OdysseusPolymetis/pyOdysseus.git

Cloning into 'pyOdysseus'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 259 (delta 25), reused 1 (delta 1), pack-reused 208 (from 1)
Receiving objects: 100% (259/259), 13.84 MiB | 23.58 MiB/s, done.
Resolving deltas: 100% (116/116), done.


In [3]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.3 MB/s eta 0:00:00


In [ ]:
%cd /content/bertalign_odysseus
!pip install .

In [ ]:
!pip install faiss-gpu-cu12

In [ ]:
!pip install numba==0.60.0 googletrans==4.0.0rc1 sentence-splitter==1.4 sentence-transformers==3.2.1

# **Alignement avec BertAlign**

In [ ]:
from bertalign import Bertalign

In [ ]:
import os
from tqdm import tqdm
from bertalign.utils import split_sents
import re
import glob
import urllib

In [ ]:
def split_chants(text):
    """
    Découpe le texte en chants.
    On suppose que chaque chant commence par une ligne contenant "Chant" suivi d’un numéro.
    Renvoie une liste de chants (chaînes de caractères).
    """
    pattern = re.compile(r'^(Chant\s*\d+)', re.IGNORECASE)
    chants = []
    current_chant = []
    for line in text.splitlines():
        line = line.strip()
        if pattern.match(line):
            if current_chant:
                chants.append("\n".join(current_chant))
                current_chant = []
        current_chant.append(line)
    if current_chant:
        chants.append("\n".join(current_chant))
    return chants

In [ ]:
def get_case_insensitive_filepath(folder, filename):
    """
    Cherche dans le dossier 'folder' un fichier dont le nom, en minuscules, correspond à 'filename' en minuscules.
    Retourne le chemin complet si trouvé, sinon None.
    """
    filename_lower = filename.lower()
    for f in os.listdir(folder):
        if f.lower() == filename_lower:
            return os.path.join(folder, f)
    return None

In [ ]:
def preprocess_chants(biblio, src_folder="/content/source", chants_folder="/content/chants"):
    """
    Pour chaque texte de la biblio, lit le fichier TXT,
    découpe en chants, et enregistre chaque chant dans un fichier nommé "textid_ChantX.txt"
    dans le dossier chants_folder.
    """
    os.makedirs(chants_folder, exist_ok=True)
    for text_id in tqdm(biblio.keys(), desc="Découpage en chants", unit="texte"):
        filename = f"{text_id}.txt"
        filepath = get_case_insensitive_filepath(src_folder, filename)
        if not filepath or not os.path.exists(filepath):
            print(f"Fichier {filepath if filepath else filename} introuvable.")
            continue
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        chants = split_chants(content)
        for i, chant in enumerate(chants, start=1):
            out_filename = os.path.join(chants_folder, f"{text_id}_Chant{i}.txt")
            with open(out_filename, "w", encoding="utf-8") as out_f:
                out_f.write(chant)
        print(f"{text_id} => {len(chants)} chants enregistrés.")

In [ ]:
biblio = {
    "bareste1842": {
        "date": 1842,
        "creator": "Eugène Bareste",
        "title": "Odyssée. Traduction nouvelle accompagnée de notes, d'explications et de commentaires",
    },
    "berard1924": {
        "date": 1924,
        "creator": "Victor Bérard",
        "title": "Les navigations d’Ulysse",
    },
    "bignan1853": {
        "date": 1841,
        "creator": "Anne Bignan",
        "title": "L’Odyssée",
    },
    "bitaube1810": {
        "date": 1785,
        "creator": "Paul Jérémie Bitaubé",
        "title": "L’Odyssée [édition de 1810]",
    },
    "boitel1619": {
        "date": 1619,
        "creator": "Boitel",
        "title": "L’Odyssée",
    },
    "calbetrosny1897": {
        "date": 1897,
        "creator": "Calbet Rosny",
        "title": "L’Odyssée",
    },
    "certon1604": {
        "date": 1604,
        "creator": "Salomon Certon",
        "title": "L’Odyssée",
    },
    "dacier1872": {
        "date": 1716,
        "creator": "Anne Dacier",
        "title": "L'Iliade et l'Odyssée. Traduction de Mme Dacier. Nouvelle édition revue, corrigée [édition de 1872]",
    },
    "dugasmontbel1833": {
        "date": 1833,
        "creator": "Jean-Baptiste Dugas-Montbel",
        "title": "L’Odyssée d’Homère",
    },
    "dufourraison1946": {
        "date": 1946,
        "creator": "Médéric Dufour et Jeanne Raison",
        "title": "L’Odyssée d’Homère",
    },
    "froment1883": {
        "date": 1883,
        "creator": "J.B.F. Froment",
        "title": "Homère",
    },
    "giguet1852": {
        "date": 1852,
        "creator": "P. Giguet",
        "title": "L’Iliade et l’Odyssée",
    },
    "hins1883": {
        "date": 1883,
        "creator": "Eugène Hins",
        "title": "L'Odyssée",
    },
    "jaccottet1955": {
        "date": 1955,
        "creator": "Philippe Jaccottet",
        "title": "L’Odyssée",
    },
    "jamyn1584": {
        "date": 1584,
        "creator": "Amadis Jamyn",
        "title": "L’Odyssée",
    },
    "lavalterie1681": {
        "date": 1681,
        "creator": "Achille de La Valterie",
        "title": "L’Odyssée d’Homère",
        "section": [1, 2, 3, 4],
    },
    "lcdlisle1867": {
        "date": 1867,
        "creator": "Leconte de Lisle",
        "title": "L’Odyssée",
    },
    "lplbrun1819": {
        "date": 1819,
        "creator": "Charles François Lebrun",
        "title": "L’Odyssée d’Homère",
    },
    "meunier1943": {
        "date": 1943,
        "creator": "Mario Meunier",
        "title": "Odyssée",
    },
    "mugler1991": {
        "date": 1991,
        "creator": "Frédéric Mugler",
        "title": "L’odyssée",
    },
    "peletierdumans1540": {
        "date": 1540,
        "creator": "Jacques Peletier du Mans",
        "title": "L’Odyssée",
    },
    "pessonneaux1866": {
        "date": 1862,
        "creator": "Émile Pessonneaux",
        "title": "L’Iliade et l’Odyssée",
    },
    "rochefort1777": {
        "date": 1777,
        "creator": "Guillaume Dubois de Rochefort",
        "title": "Odyssée",
    },
    "seguier1896": {
        "date": 1896,
        "creator": "Ulysse de Séguier",
        "title": "L'Odyssée",
    },
    "sommer1886": {
        "date": 1886,
        "creator": "Édouard Sommer",
        "title": "L’Odyssée en juxtalinéaire",
    },
}

In [ ]:
preprocess_chants(biblio, src_folder="/content/source", chants_folder="/content/chants")

In [ ]:
def find_sommer_pivot(biblio_dict):
    """
    Retourne l'identifiant du texte dont le nom ou le titre contient 'sommer' (en minuscule).
    En l'occurrence, le pivot choisi contient sommer, mais on peut changer.
    """
    for text_id, meta in biblio_dict.items():
        if ("sommer" in text_id.lower()) or ("sommer" in meta.get("title", "").lower()):
            return text_id
    return None

In [ ]:
def read_raw_text(text_id, folder="raw_texts"):
    """
    Lit le fichier brut <text_id>.txt dans le dossier raw_texts/
    et renvoie le contenu en une seule chaîne.
    """
    filename = os.path.join(folder, f"{text_id}.txt")
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Fichier introuvable : {filename}")
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()

In [ ]:
def align_pairwise(src_text, tgt_text):
    """
    Aligne deux chaînes (texte source et texte cible)
    via BERTAlign (modifié), qui réalise sa propre segmentation.
    Retourne l'objet Bertalign (avec alignments dans .result).
    """
    aligner = Bertalign(src_text, tgt_text)
    alignment_result = aligner.align_sents()
    return alignment_result

In [ ]:
def extract_alignment_map(alignment_result):
    """
    Extrait l'alignement final M↔N depuis alignment_result.result,
    qui est une liste de tuples (src_range, tgt_range).

    - src_range : liste des indices source (pivot)
    - tgt_range : liste des indices cible

    Construit un dict { i_src: [liste_indices_cibles] }.

    Exemple:
      alignment_result.result = [
        ([0], [4,5]),
        ([1,2], [6]),
        ([3], [7,8]),
        ...
      ]
    On veut:
      mapping[0] = [4,5]
      mapping[1] = [6]
      mapping[2] = [6]
      mapping[3] = [7,8]
      etc.
    """
    final_list = getattr(alignment_result, "result", [])
    if not final_list:
        return {}

    mapping = {}
    for (src_range, tgt_range) in final_list:
        for s in src_range:
            if s not in mapping:
                mapping[s] = []
            for t in tgt_range:
                if t not in mapping[s]:
                    mapping[s].append(t)
    for s in mapping:
        mapping[s].sort()

    return mapping

In [ ]:
def build_multi_alignment(biblio_dict, chant_number, folder="/content/chants"):
    pivot_id = find_sommer_pivot(biblio_dict)
    if not pivot_id:
        raise ValueError("Aucun texte 'sommer' trouvé comme pivot.")

    pivot_filepath = os.path.join(folder, f"{pivot_id}_Chant{chant_number}.txt")
    if not os.path.exists(pivot_filepath):
        raise ValueError(f"Fichier pivot introuvable : {pivot_filepath}")
    with open(pivot_filepath, "r", encoding="utf-8") as f:
        pivot_text = f.read()
    pivot_sents = split_sents(pivot_text, "fr")
    nb_pivot_sents = len(pivot_sents)
    print(f"Pivot {pivot_id} détecté avec {nb_pivot_sents} phrases (segmentation BERTAlign).")

    other_text_ids = [tid for tid in biblio_dict.keys() if tid != pivot_id]
    pairwise_alignments = {}
    for tid in tqdm(other_text_ids, desc="Aligning pivot with other texts", unit="text"):
        tgt_filepath = os.path.join(folder, f"{tid}_Chant{chant_number}.txt")
        if not os.path.exists(tgt_filepath):
            print(f"Fichier introuvable pour {tid}: {tgt_filepath}")
            continue
        with open(tgt_filepath, "r", encoding="utf-8") as f:
            tgt_text = f.read()
        result = align_pairwise(pivot_text, tgt_text)
        alignment_map = extract_alignment_map(result)
        pairwise_alignments[tid] = alignment_map

        nb_aligned_segments = sum(1 for v in alignment_map.values() if v)
        print(f"\n[DEBUG] => Alignement pivot={pivot_id} vs {tid}:")
        print(f"   - Nombre de segments du pivot réellement alignés: {nb_aligned_segments} / {nb_pivot_sents}")
        sample_keys = list(alignment_map.keys())[:5]
        for k in sample_keys:
            print(f"   * pivot segment {k} => cibles {alignment_map[k]}")
        print("-" * 60)

    print("\n[DEBUG] Vérification immédiate des segments pivot sans alignement :")
    for i in range(nb_pivot_sents):
        missing = [tid for tid in other_text_ids if i not in pairwise_alignments.get(tid, {}) or not pairwise_alignments[tid][i]]
        if missing:
            print(f"  - Pivot segment {i} : '{pivot_sents[i]}' -> aucun alignement pour : {', '.join(missing)}")
    print("-" * 60)

    used = {tid: set() for tid in other_text_ids}
    multi_alignment = []
    for i in tqdm(range(nb_pivot_sents), desc="Building multi-alignment", unit="segment"):
        row = {pivot_id: i}
        for tid in other_text_ids:
            raw_indices = pairwise_alignments.get(tid, {}).get(i, [])
            if isinstance(raw_indices, int):
                raw_indices = [raw_indices]
            new_indices = []
            seen = set()
            for idx in raw_indices:
                if idx not in seen and idx not in used[tid]:
                    seen.add(idx)
                    new_indices.append(idx)
            row[tid] = new_indices
            used[tid].update(new_indices)
        multi_alignment.append(row)
    return pivot_id, multi_alignment, pairwise_alignments, pivot_sents

In [ ]:
def build_multi_alignment_chant(biblio, chant_number, chants_folder="/content/chants"):
    """
    Pour un numéro de chant donné, lit pour chaque texte le fichier de chant correspondant,
    utilise build_all_sents_chant pour obtenir la segmentation en phrases,
    puis reconstruit temporairement des textes complets (en joignant les segments) afin
    de lancer build_multi_alignment.
    Renvoie pivot_id, multi_aligned, et all_sents (pour ce chant).
    """
    all_sents = build_all_sents_chant(biblio, chant_number, chants_folder)
    pivot_id = find_sommer_pivot(biblio)
    if pivot_id not in all_sents:
        raise ValueError(f"Le pivot {pivot_id} n'a pas de chant {chant_number}.")
    temp_folder = "/tmp/chants_for_alignment"
    os.makedirs(temp_folder, exist_ok=True)
    for tid, segments in all_sents.items():
        temp_path = os.path.join(temp_folder, f"{tid}.txt")
        with open(temp_path, "w", encoding="utf-8") as f:
            f.write("\n".join(segments))
    pivot_id, multi_aligned = build_multi_alignment(biblio, folder=temp_folder)
    return pivot_id, multi_aligned, all_sents

In [ ]:
def build_all_sents_chant(biblio, chant_number, chants_folder="/content/chants"):
    """
    Pour un numéro de chant donné, lit pour chaque texte le fichier correspondant
    (par exemple, "Sommer1886_Chant2.txt") depuis le dossier chants.
    Utilise split_sents pour découper le contenu en phrases (dans bertalign).
    Renvoie un dict all_sents où all_sents[text_id] est la liste des segments (phrases) du chant.
    """
    all_sents = {}
    for text_id in biblio.keys():
        filepath = os.path.join(chants_folder, f"{text_id}_Chant{chant_number}.txt")
        if not os.path.exists(filepath):
            print(f"Fichier introuvable : {filepath}")
            continue
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
            segments = split_sents(content, "fr")
            all_sents[text_id] = segments
            print(f"{text_id} Chant{chant_number} : {len(segments)} segments")
    return all_sents

# **Calcul des fréquences**

In [ ]:
!wget https://raw.githubusercontent.com/OdysseusPolymetis/colabs_for_nlp/refs/heads/main/stopwords_fr.txt -P /content/
text = open("/content/stopwords_fr.txt", "r").read()
STOPWORDS = text.split("\n")

In [ ]:
!pip install stanza

In [ ]:
import stanza
from collections import Counter

stanza.download("fr")

nlp = stanza.Pipeline("fr", processors="tokenize,pos,lemma")

In [ ]:
def lemmatize_phrase(phrase, nlp):
    """
    Lemmatiser une phrase à la volée via la pipeline Stanza.
    Retourne une liste de tuples (forme, lemme).
    """
    doc = nlp(phrase)
    tokens = []
    for sent in doc.sentences:
        for token in sent.tokens:
            for w in token.words:
                tokens.append((w.text, w.lemma))
    return tokens

In [ ]:
def compute_local_lemma_freq_and_authors(row, all_sents, nlp, all_tokens=None):
    freq = Counter()
    lemma_authors = {}
    for text_id, indices in row.items():
        if isinstance(indices, int):
            indices = [indices]
        if not indices:
            continue
        for idx in indices:
            if text_id not in all_sents:
                continue
            if idx >= len(all_sents[text_id]):
                print(f"[WARNING] Pour {text_id}, indice {idx} hors de portée (max {len(all_sents[text_id])}).")
                continue
            if all_tokens is not None and text_id in all_tokens and idx < len(all_tokens[text_id]):
                tokens = all_tokens[text_id][idx]
            else:
                phrase = all_sents[text_id][idx]
                tokens = lemmatize_phrase(phrase, nlp)
            for forme, lemma in tokens:
                freq[lemma] += 1
                if lemma not in lemma_authors:
                    lemma_authors[lemma] = set()
                lemma_authors[lemma].add(text_id)
    return freq, lemma_authors

In [ ]:
def new_get_freq_class(lemma, local_counter, lemma_authors, n_authors):
    """
    Retourne la classe (chaine : "freq0", "freq1", "freq2", "freq3", "freq4" ou "freq5")
    selon les critères suivants :
      - Si lemma est dans STOPWORDS → freq0 (noir)
      - Si le lemme apparaît seulement dans le segment en cours (local_counter == 1) → freq1 (rouge, low)
      - Sinon, on considère le nombre d’auteurs (global) dans lesquels il apparaît :
          * s'il apparaît chez moins d'un quart des auteurs → freq2 (rouge foncé, plag)
          * s'il apparaît chez tous, ou tous sauf deux (n_authors - 2 ou plus) → freq5
          * s'il apparaît chez au moins 70% des auteurs mais pas dans la condition précédente → freq4
          * Sinon → freq3 (neutre)
    """
    if lemma in STOPWORDS:
        return "freq0"
    if local_counter.get(lemma, 0) == 1:
        return "freq1"
    authors_count = len(lemma_authors.get(lemma, []))
    if authors_count < n_authors / 4:
        return "freq2"
    if authors_count >= 0.7 * n_authors:
        if authors_count >= n_authors - 2:
            return "freq5"
        else:
            return "freq4"
    return "freq3"

In [ ]:
def render_segment(text_id, idx, all_sents, local_counter, n_authors, nlp, lemma_authors, all_tokens=None):
    """
    Génère le HTML pour le segment (phrase) d'un texte donné en utilisant
    new_get_freq_class pour déterminer la classe CSS de chaque token.
    """
    if text_id not in all_sents or idx >= len(all_sents[text_id]):
        return f"<span style='color:red;'>(Segment {idx} introuvable pour {text_id})</span>"
    if all_tokens is not None and text_id in all_tokens and idx < len(all_tokens[text_id]):
        tokens = all_tokens[text_id][idx]
    else:
        phrase = all_sents[text_id][idx]
        tokens = lemmatize_phrase(phrase, nlp)
    token_html = []
    for forme, lemma in tokens:
        cls = new_get_freq_class(lemma, local_counter, lemma_authors, n_authors)
        tooltip = ""
        if cls != "freq0" and local_counter.get(lemma, 0) <= 1 and lemma in lemma_authors:
            tooltip = " title='Found in: " + ", ".join(sorted(lemma_authors[lemma])) + "'"
        if cls == "freq2":
            inner = f'<a title="[{", ".join(sorted(lemma_authors.get(lemma, [])))}]">{forme}</a>'
            token_html.append(f'<mark class="{cls}">{inner}</mark>')
        else:
            token_html.append(f'<mark class="{cls}"{tooltip}>{forme}</mark>')
    return " ".join(token_html)

In [ ]:
def format_stats(stats):
    high = stats.get("high", "N/A")
    low = stats.get("low", "N/A")
    mid = stats.get("mid", "N/A")
    try:
        return f"Hautes: {float(high):.0%} | Faibles: {float(low):.0%} | Basses: {float(mid):.0%}"
    except (ValueError, TypeError):
        return f"Hautes: {high} | Faibles: {low} | Basses: {mid}"


In [ ]:
def compute_stats_per_author(biblio, multi_alignment, all_tokens):
    """
    Pour un chant donné, parcourt le multi_alignment et les all_tokens pour construire :
      - un compteur de tokens par auteur (per_author_counter)
      - un compteur global (global_counter)
      - un mapping global des auteurs pour chaque lemme (global_authors)
    Puis, pour chaque auteur, calcule les ratios de tokens en catégorie "low" (freq1), "plag" (freq2) et "high" (freq5)
    et, pour les tokens en plag, rassemble un compteur des auteurs partenaires (pour voir avec qui ils sont partagés).
    """
    per_author_counter = {tid: Counter() for tid in biblio.keys()}
    for row in multi_alignment:
        for tid in biblio.keys():
            indices = row.get(tid, [])
            if isinstance(indices, int):
                indices = [indices]
            for idx in indices:
                if tid in all_tokens and idx < len(all_tokens[tid]):
                    tokens = all_tokens[tid][idx]
                    for forme, lemma in tokens:
                        per_author_counter[tid][lemma] += 1

    global_counter = Counter()
    global_authors = {}
    for tid, counter in per_author_counter.items():
        for lemma, count in counter.items():
            global_counter[lemma] += count
            if lemma not in global_authors:
                global_authors[lemma] = set()
            if count > 0:
                global_authors[lemma].add(tid)

    n_authors = len(biblio)
    stats = {}
    for tid, counter in per_author_counter.items():
        total = sum(counter.values())
        low = sum(count for lemma, count in counter.items() if new_get_freq_class(lemma, global_counter, global_authors, n_authors) == "freq1")
        plag = sum(count for lemma, count in counter.items() if new_get_freq_class(lemma, global_counter, global_authors, n_authors) == "freq2")
        high = sum(count for lemma, count in counter.items() if new_get_freq_class(lemma, global_counter, global_authors, n_authors) == "freq5")

        plag_authors = Counter()
        for lemma, count in counter.items():
            if new_get_freq_class(lemma, global_counter, global_authors, n_authors) == "freq2":
                for other in global_authors.get(lemma, set()):
                    if other != tid:
                        plag_authors[other] += count
        stats[tid] = {
            "total": total,
            "low_ratio": low / total if total > 0 else 0,
            "plag_ratio": plag / total if total > 0 else 0,
            "high_ratio": high / total if total > 0 else 0,
            "plag_authors": plag_authors,
        }
    return stats

In [ ]:
def generate_section_html(text_id, pivot_id, chant_number, multi_alignment, all_sents, nlp, all_tokens=None, title_attr=None, n_authors=1):
    section_global_id = f"{text_id.lower()}_{chant_number:02d}"
    title_attr = title_attr or f"{text_id} Chant {chant_number}"
    lines = [f'<section id="{section_global_id}" class="parallel" title="{title_attr}">']

    for block_index, row in enumerate(multi_alignment, start=1):
        pivot_idx = row.get(pivot_id)
        if pivot_idx is None or pivot_idx >= len(all_sents.get(pivot_id, [])):
            pivot_text = "(Segment pivot introuvable)"
        else:
            pivot_text = all_sents[pivot_id][pivot_idx]
        indices = row.get(text_id, [])
        if isinstance(indices, int):
            indices = [indices]
        if not indices:
            seg_html = "∅"
        else:
            local_counter, local_lemma_authors = compute_local_lemma_freq_and_authors(row, all_sents, nlp, all_tokens)
            segs = []
            for idx in indices:
                if idx < len(all_sents.get(text_id, [])):
                    html_segment = render_segment(text_id, idx, all_sents, local_counter, n_authors, nlp, local_lemma_authors, all_tokens)
                    segs.append(html_segment if html_segment.strip() else "∅")
                else:
                    segs.append(f"(Segment {idx} introuvable)")
            seg_html = " ".join(segs)
        lines.append(f"<div class='chunk syn{pivot_idx if pivot_idx is not None else 0}' id='{section_global_id}-{block_index}'><b>{block_index} </b>{seg_html}</div>")
    lines.append("</section>")
    return "\n".join(lines)

In [ ]:
def generate_html_per_author(biblio, chant_number, multi_alignment, all_sents, nlp, all_tokens=None, output_dir="/content/aligned", pairwise_alignments=None):
    os.makedirs(output_dir, exist_ok=True)
    pivot_id = find_sommer_pivot(biblio)
    n_authors = len(biblio)

    # Calcul des stats par auteur pour ce chant
    stats = compute_stats_per_author(biblio, multi_alignment, all_tokens)

    for text_id, stat in stats.items():
        if stat["total"] == 0:
            print(f"Aucun alignement pour {text_id} sur le chant {chant_number}.")
            continue
        plag_shared = ", ".join(f"{other}({cnt})" for other, cnt in stat["plag_authors"].most_common(2))
        title_attr = f"{text_id}: low {stat['low_ratio']:.0%}, plag {stat['plag_ratio']:.0%}, high {stat['high_ratio']:.0%}"
        if plag_shared:
            title_attr += f" (plag partagé avec: {plag_shared})"
        section_html = generate_section_html(text_id, pivot_id, chant_number, multi_alignment, all_sents, nlp, all_tokens, title_attr=title_attr, n_authors=n_authors)
        output_file = os.path.join(output_dir, f"{text_id.lower()}_{chant_number}.html")
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(section_html)
        print(f"HTML généré pour {text_id} Chant {chant_number} dans {output_file}")


In [ ]:
ref_text = list(biblio.keys())[0]
chant_files = glob.glob(os.path.join("/content/chants", f"{ref_text}_Chant*.txt"))
chant_numbers = sorted([int(re.search(r'Chant(\d+)', os.path.basename(f)).group(1)) for f in chant_files])

for chant_number in chant_numbers:
    print(f"\n=== Traitement du Chant {chant_number} ===")

    # Lire les chants pour chaque texte
    all_sents = {}
    for text_id in biblio.keys():
        filepath = os.path.join("/content/chants", f"{text_id}_Chant{chant_number}.txt")
        if not os.path.exists(filepath):
            print(f"Fichier introuvable : {filepath}")
            continue
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        all_sents[text_id] = split_sents(content, "fr")

    # Construire l'alignement pour ce chant
    pivot_id, multi_alignment, pairwise_alignments, pivot_sents = build_multi_alignment(biblio, chant_number, folder="/content/chants")

    # Générer all_tokens pour ce chant
    all_tokens = {}
    for text_id, segments in all_sents.items():
        tokens_list = []
        for segment in segments:
            tokens_list.append(lemmatize_phrase(segment, nlp))
        all_tokens[text_id] = tokens_list

    # Générer le HTML par auteur et calculer les stats par auteur pour ce chant
    generate_html_per_author(biblio,
                             chant_number=chant_number,
                             multi_alignment=multi_alignment,
                             all_sents=all_sents,
                             nlp=nlp,
                             all_tokens=all_tokens,
                             output_dir="/content/aligned",
                             pairwise_alignments=pairwise_alignments)

    # Libérer les variables
    del all_sents, multi_alignment, pairwise_alignments, pivot_sents, all_tokens

    print(f"=== Chant {chant_number} traité et HTML généré ===")

Aligning pivot with other texts:  42%|████▏     | 10/24 [02:03<02:56, 12.63s/text]

Performing first-step alignment ...
Performing second-step alignment ...
Finished! Successfully aligning 191 French sentences to 254 French sentences


[DEBUG] => Alignement pivot=sommer1886 vs dufourraison1946:
   - Nombre de segments du pivot réellement alignés: 190 / 191
   * pivot segment 0 => cibles [0]
   * pivot segment 1 => cibles [1, 2]
   * pivot segment 2 => cibles [3]
   * pivot segment 3 => cibles [3]
   * pivot segment 4 => cibles [4, 5]
------------------------------------------------------------
Source language: French, Number of sentences: 191
Target language: French, Number of sentences: 583
Embedding source and target text using LaBSE ...


Aligning pivot with other texts:  46%|████▌     | 11/24 [02:15<02:42, 12.48s/text]

Performing first-step alignment ...
Performing second-step alignment ...
Finished! Successfully aligning 191 French sentences to 583 French sentences


[DEBUG] => Alignement pivot=sommer1886 vs froment1883:
   - Nombre de segments du pivot réellement alignés: 191 / 191
   * pivot segment 0 => cibles [0]
   * pivot segment 1 => cibles [1, 2, 3]
   * pivot segment 2 => cibles [4, 5]
   * pivot segment 3 => cibles [4, 5]
   * pivot segment 4 => cibles [6, 7, 8, 9]
------------------------------------------------------------
Source language: French, Number of sentences: 191
Target language: French, Number of sentences: 253
Embedding source and target text using LaBSE ...


In [ ]:
# ça c'est si on veut après faire le transfert des résultats sur le drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
